# Recommendation / 02 — Preprocessing

Run crowd notebooks first. Recommendation preprocessing needs the selected crowd model.

Run every cell from top to bottom. Use the project `.venv` kernel in VS Code. No season field is used. Data and test scores are synthetic.

In [1]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml_service/workflows.py").exists())
sys.path.insert(0, str(ROOT))
from ml_service import workflows as w
print("Project:", ROOT)
print("Synthetic demonstration: real-world accuracy has not been measured.")

Project: /workspace/scratch/71e3a62599e6/YatraSense_Organic
Synthetic demonstration: real-world accuracy has not been measured.


In [2]:
print(inspect.getsource(w.preprocess_recommendation))
print(inspect.getsource(w.preprocessing))

def preprocess_recommendation():
    original=pd.read_csv(DATA/'ingested.csv')
    crowd=pd.read_csv(DATA/'crowd.csv'); train_frame=crowd[crowd.dataset_split=='train'].copy()
    crowd_model=joblib.load(MODELS/'crowd.joblib')
    # Grouped out-of-fold predictions prevent feeding memorized crowd labels
    # into the recommendation training examples. Group entire years.
    oof=cross_val_predict(clone(crowd_model),train_frame[CROWD_FEATURES],train_frame.busyness_label,cv=GroupKFold(n_splits=4),groups=train_frame.year,n_jobs=1)
    predictions=dict(zip(train_frame.destination_month_id,oof))
    held=crowd[crowd.dataset_split!='train']
    predictions.update(dict(zip(held.destination_month_id,crowd_model.predict(held[CROWD_FEATURES]))))
    original['estimated_busyness']=original.destination_month_id.map(predictions)
    df=rec_features(original)
    assert df.estimated_busyness.notna().all()
    df.to_csv(DATA/'recommendation.csv',index=False)
    return {'rows':len(df),'crowd_features':

In [3]:
summary = w.preprocess_recommendation()
print(summary)
df = pd.read_csv(w.DATA / "recommendation.csv")
display(df.head())

{'rows': 14400, 'crowd_features': 'grouped out-of-fold for train; frozen train-only model for validation/test', 'features': ['destination_id', 'area', 'popularity_score', 'temperature_mean_c', 'precipitation_mm_month', 'wind_speed_kmh_mean', 'month_sin', 'month_cos', 'heritage_pilgrimage', 'trekking_hiking', 'wildlife_birdwatching', 'boating_water', 'adventure_sports', 'customer_heritage_pilgrimage', 'customer_trekking_hiking', 'customer_wildlife_birdwatching', 'customer_boating_water', 'customer_adventure_sports', 'activity_match', 'weather_comfort', 'customer_crowd_preference', 'estimated_busyness', 'crowd_match']}


,row_id,destination_month_id,request_id,destination_id,destination_name,area,destination_type,year,month,travel_date,temperature_mean_c,precipitation_mm_month,wind_speed_kmh_mean,weather_kind,popularity_score,popularity_level,heritage_pilgrimage,trekking_hiking,wildlife_birdwatching,boating_water,adventure_sports,specific_activities,visitor_count,count_scope,busyness_label,customer_heritage_pilgrimage,customer_trekking_hiking,customer_wildlife_birdwatching,customer_boating_water,customer_adventure_sports,customer_crowd_preference,customer_chose_destination,dataset_split,is_synthetic,metadata_status,record_status,source_url,generation_version,estimated_busyness,month_sin,month_cos,activity_match,weather_comfort,crowd_match
0,ROW1,YS001-2016-01,R201601-1,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,0,0,1,0,0,Medium,1,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930,Medium,0.5,0.866025,0.0,0.922769,1
1,ROW2,YS001-2016-01,R201601-2,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,1,0,0,1,0,High,0,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930,Medium,0.5,0.866025,0.5,0.922769,0
2,ROW3,YS001-2016-01,R201601-3,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,0,1,0,1,0,Any,1,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930,Medium,0.5,0.866025,0.0,0.922769,1
3,ROW4,YS001-2016-01,R201601-4,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,1,0,0,1,0,Low,0,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930,Medium,0.5,0.866025,0.5,0.922769,0
4,ROW5,YS001-2016-01,R201601-5,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,0,1,1,0,1,Medium,0,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930,Medium,0.5,0.866025,0.0,0.922769,1


Numeric imputation and scaling, plus categorical imputation and one-hot encoding, are fitted inside each training pipeline. Validation and test records never fit those transforms. Recommendation crowd features are grouped out-of-fold predictions for training and frozen train-only predictions for later years. Those folds group whole years; they are cross-fitting, not a rolling historical backtest.